In [ ]:
# parameters
# BINDER_FAST: set N=6 for fast cloud execution
N = 20    # Hilbert space truncation
Ej = 40.0  # Josephson energy (GHz), deep transmon: Ej/Ec = 200
Ec = 0.2   # Charging energy (GHz)


In [ ]:
# hide
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
%matplotlib widget

from bosonic_gates.hamiltonians.transmon import transmon_hamiltonian


## Module 1b: The Transmon Hamiltonian

**Learning objectives:**
- Expand $\cos(\hat{\phi})$ in ladder operators and identify the Kerr nonlinearity
- Compute the transmon transition frequencies and anharmonicity numerically
- Understand charge dispersion and its exponential suppression in the transmon limit
- Relate the Kerr constant $K$ to the measurable anharmonicity $\alpha$

---

**Sections:** [1 Kerr Nonlinearity](#sec1) · [2 Transmon Spectrum](#sec2) · [3 Charge Dispersion](#sec3) · [4 Transmon Limit](#sec4)


<a id="sec1"></a>
## 1  The Kerr Nonlinearity from $\cos(\hat{\phi})$

The full Cooper-pair box Hamiltonian is

$$\hat{H} = 4E_C(\hat{n} - n_g)^2 - E_J\cos(\hat{\phi}).$$

In the transmon limit ($E_J/E_C \gg 1$), phase fluctuations are small and we can expand the cosine:

$$\cos(\hat{\phi}) = 1 - \frac{\hat{\phi}^2}{2} + \frac{\hat{\phi}^4}{24} - \frac{\hat{\phi}^6}{720} + \cdots$$

**Phase operator in terms of ladder operators.** Introduce the zero-point phase fluctuation
$\phi_0 = (8E_C/E_J)^{1/4}$; then

$$\hat{\phi} = \frac{\phi_0}{\sqrt{2}}(\hat{a} + \hat{a}^\dagger).$$

Substituting and normal-ordering:

$$\hat{H} \approx \omega_{01}\hat{a}^\dagger\hat{a} - \frac{K}{2}\hat{a}^{\dagger 2}\hat{a}^2,$$

where the **Kerr constant** is

$$K = E_C$$

(to leading order in $\phi_0^2$).
The second term $-\frac{K}{2}\hat{n}(\hat{n}-1)$ shifts the $n$-photon level down by $Kn(n-1)/2$,
breaking the equal spacing of the harmonic oscillator.

**Anharmonicity.** The transition frequencies are

$$\omega_{01} = E_1 - E_0, \qquad \omega_{12} = E_2 - E_1 = \omega_{01} - K.$$

The anharmonicity $\alpha \equiv \omega_{12} - \omega_{01} = -K \approx -E_C < 0$.

This **negative anharmonicity** is the defining feature of the transmon: the $|1\rangle \to |2\rangle$
transition is red-detuned from the $|0\rangle \to |1\rangle$ transition by $|E_C|$.
A resonant drive on $\omega_{01}$ will off-resonantly drive $\omega_{12}$ with an error
scaling as $(\Omega/\alpha)^2$ where $\Omega$ is the drive Rabi rate.
Typical $|\alpha|/2\pi \approx 150$–$300\,\text{MHz}$ limits gates to $\sim 20$–$50\,\text{ns}$.

> **Lab note.** *The Kerr constant $K$ is directly measurable by two-tone spectroscopy:
> drive the qubit at $\omega_{01}$ and probe $\omega_{12}$.
> The frequency difference is $|\alpha| = K$.
> In the rotating frame, the Kerr term appears as a photon-number-dependent frequency shift,
> which is also how the qubit reads out the cavity state in dispersive coupling (Module 1c).*


In [ ]:
# Compute exact omega_01 and anharmonicity for default parameters
H = transmon_hamiltonian(Ej, Ec, N)
evals = H.eigenenergies()[:5]

omega01 = (evals[1] - evals[0]) / (2 * np.pi)
omega12 = (evals[2] - evals[1]) / (2 * np.pi)
alpha   = omega12 - omega01

print(f"Transmon parameters: Ej = {Ej} GHz, Ec = {Ec} GHz, Ej/Ec = {Ej/Ec:.0f}")
print(f"  omega_01 / 2pi = {omega01:.4f} GHz")
print(f"  omega_12 / 2pi = {omega12:.4f} GHz")
print(f"  anharmonicity alpha / 2pi = {alpha:.4f} GHz")
print(f"  -Ec = {-Ec:.4f} GHz  (leading-order Kerr prediction)")
print(f"  deviation from -Ec: {abs(alpha - (-Ec)):.4f} GHz  ({abs(alpha - (-Ec))/Ec*100:.2f}%)")

# Analytic estimate from Koch et al.
omega01_approx = np.sqrt(8 * Ej * Ec) - Ec
print(f"\nAnalytic omega_01 ~ sqrt(8*Ej*Ec) - Ec = {omega01_approx:.4f} GHz  "
      f"(error = {abs(omega01 - omega01_approx):.4f} GHz)")


<a id="sec2"></a>
## 2  The Transmon Spectrum

Diagonalizing the full transmon Hamiltonian exactly (no truncation of the cosine) gives
a spectrum that is accurately described by the Kerr model for the lowest levels,
but deviates at higher levels where the cosine potential turns over.

The $n$-th level (in the Kerr approximation) has energy

$$E_n \approx n\,\omega_{01} - \frac{K}{2}n(n-1),$$

so the transition frequency for $|n-1\rangle \to |n\rangle$ is

$$\omega_{n-1,n} = \omega_{01} - (n-1)K.$$

The spectrum fans out linearly: each successive rung is $K$ lower than the previous one.
In the bar diagram below you can see this fanning pattern clearly.

> **Lab note.** *In a typical cQED experiment, you only use $|0\rangle$ and $|1\rangle$ as the qubit
> computational basis.
> However, higher levels can be accessed deliberately for qutrit gates or SNAP protocols (Module 4).
> Leakage to $|2\rangle$ during gates is the main error for short pulses — the anharmonicity sets
> a fundamental limit on gate speed via $t_{\rm gate} \gtrsim \pi/|\alpha|$.*


In [ ]:
# Transmon spectrum: bar diagram of first 6 levels with transition annotations
n_show = 6
H = transmon_hamiltonian(Ej, Ec, N)
evals = H.eigenenergies()[:n_show]
evals_shifted = evals - evals[0]  # zero at ground state

# Kerr approximation: E_n = n*omega01 - K/2 * n*(n-1)
K_approx = Ec  # GHz
omega01_approx = np.sqrt(8 * Ej * Ec) - Ec  # GHz
n_arr = np.arange(n_show)
E_kerr = (n_arr * omega01_approx - 0.5 * K_approx * n_arr * (n_arr - 1)) * 2 * np.pi

fig, ax = plt.subplots(figsize=(5, 7))

colors_level = plt.cm.Blues(np.linspace(0.4, 0.9, n_show))

for n, (E, Ek, c) in enumerate(zip(evals_shifted, E_kerr, colors_level)):
    E_ghz = E / (2 * np.pi)
    Ek_ghz = Ek / (2 * np.pi)
    ax.barh(E_ghz, 0.6, left=0.2, height=0.04 * evals_shifted[-1] / (2 * np.pi),
            color=c, edgecolor="navy", linewidth=0.8)
    ax.text(0.88, E_ghz, rf"$|{n}\rangle$", va="center", fontsize=11,
            transform=ax.get_yaxis_transform())
    if n > 0:
        prev_E_ghz = evals_shifted[n - 1] / (2 * np.pi)
        trans_ghz = E_ghz - prev_E_ghz
        mid = (E_ghz + prev_E_ghz) / 2
        ax.annotate("", xy=(0.13, E_ghz), xytext=(0.13, prev_E_ghz),
                    xycoords=("axes fraction", "data"),
                    textcoords=("axes fraction", "data"),
                    arrowprops=dict(arrowstyle="<->", color="tomato", lw=1.5))
        ax.text(0.02, mid, rf"$\omega_{{{n-1},{n}}}/2\pi$" + f"\n= {trans_ghz:.3f} GHz",
                va="center", ha="left", fontsize=7.5, color="tomato",
                transform=ax.get_yaxis_transform())

ax.set_ylabel(r"Energy $/ 2\pi$ (GHz)", fontsize=12)
ax.set_xticks([])
ax.set_title(
    rf"Transmon spectrum: $E_J = {Ej}$, $E_C = {Ec}$ GHz, $E_J/E_C = {Ej/Ec:.0f}$",
    fontsize=10
)
ax.spines[["top", "right", "bottom"]].set_visible(False)
plt.tight_layout()
plt.show()

print("Level energies (GHz) — exact vs Kerr approximation:")
print(f"{'n':>3}  {'E_exact':>10}  {'E_Kerr':>10}  {'diff':>10}")
for n in range(n_show):
    E_ex  = evals_shifted[n] / (2 * np.pi)
    E_k   = E_kerr[n] / (2 * np.pi)
    print(f"{n:>3}  {E_ex:>10.4f}  {E_k:>10.4f}  {E_ex - E_k:>10.4f}")


<a id="sec3"></a>
## 3  Charge Dispersion

The charge dispersion $\epsilon_m$ of the $m$-th level is defined as the peak-to-peak variation of
$E_m(n_g)$ as the gate charge $n_g$ is swept over one period:

$$\epsilon_m = \max_{n_g} E_m(n_g) - \min_{n_g} E_m(n_g).$$

Koch et al. (2007) derived the asymptotic formula

$$\epsilon_m \approx (-1)^m E_C \frac{2^{4m+5}}{m!} \sqrt{\frac{2}{\pi}}\left(\frac{E_J}{2E_C}\right)^{m/2 + 3/4}
\exp\!\left(-\sqrt{8E_J/E_C}\right).$$

The key observation is the **exponential suppression** $\propto e^{-\sqrt{8E_J/E_C}}$.
For the qubit level ($m=1$), this means:

| $E_J/E_C$ | $e^{-\sqrt{8E_J/E_C}}$ |
|-----------|---------------------|
| 1 | $e^{-2.83} \approx 0.06$ |
| 10 | $e^{-8.94} \approx 1.3 \times 10^{-4}$ |
| 50 | $e^{-20.0} \approx 2.1 \times 10^{-9}$ |
| 100 | $e^{-28.3} \approx 4.9 \times 10^{-13}$ |

By $E_J/E_C \approx 50$ the charge dispersion is below the $1\,\text{Hz}$ level — completely negligible
compared to other decoherence sources.

> **Lab note.** *You can measure $\epsilon_1$ by applying a DC voltage to a gate electrode and
> watching the qubit frequency shift in two-tone spectroscopy.
> For a well-tuned transmon with $E_J/E_C \approx 50$, the shift is typically $< 1\,\text{kHz}$ per $e$
> of gate charge — buried under the $T_2$ linewidth.*


In [ ]:
# Charge dispersion: E_1(ng) - E_1(0) for several Ej/Ec values
ng_vals = np.linspace(0, 0.5, 50)  # half period (symmetric)

ratios_disp = [1, 5, 20, 50]
Ec_disp = 0.2  # GHz fixed
colors_disp = plt.cm.plasma(np.linspace(0.1, 0.85, len(ratios_disp)))

fig, ax = plt.subplots(figsize=(8, 5))

for ratio, c in zip(ratios_disp, colors_disp):
    Ej_disp = ratio * Ec_disp
    E1_vals = []
    for ng in ng_vals:
        H = transmon_hamiltonian(Ej_disp, Ec_disp, N, ng=ng)
        evals = H.eigenenergies()[:3]
        E1_vals.append(evals[1])
    E1_arr = np.array(E1_vals)
    E1_shifted = (E1_arr - E1_arr[0]) / (2 * np.pi)  # GHz, zero at ng=0
    ax.semilogy(ng_vals, np.abs(E1_shifted) + 1e-14, color=c, lw=2,
                label=rf"$E_J/E_C = {ratio}$")

ax.set_xlabel(r"Gate charge $n_g$", fontsize=12)
ax.set_ylabel(r"$|E_1(n_g) - E_1(0)|\;/ 2\pi$ (GHz)", fontsize=12)
ax.set_title(r"Charge dispersion vs $n_g$ for several $E_J/E_C$ (log scale)", fontsize=11)
ax.legend(fontsize=10)
ax.set_xlim(0, 0.5)
plt.tight_layout()
plt.show()

# Print peak-to-peak dispersions
print("Peak-to-peak charge dispersion of qubit level:")
for ratio in ratios_disp:
    Ej_disp = ratio * Ec_disp
    E1_vals = []
    for ng in np.linspace(0, 0.5, 80):
        H = transmon_hamiltonian(Ej_disp, Ec_disp, N, ng=ng)
        evals = H.eigenenergies()[:2]
        E1_vals.append(evals[1])
    E1_arr = np.array(E1_vals)
    eps1 = (E1_arr.max() - E1_arr.min()) / (2 * np.pi)
    # Koch formula prediction
    eps1_approx = Ec_disp * np.sqrt(2/np.pi) * (32 * Ej_disp / (2 * Ec_disp))**0.75 * \
                  np.exp(-np.sqrt(8 * Ej_disp / Ec_disp)) * 2**5
    print(f"  Ej/Ec = {ratio:>3}: epsilon_1 = {eps1:.2e} GHz  "
          f"(Koch formula: {abs(eps1_approx):.2e} GHz)")


<a id="sec4"></a>
## 4  The Transmon Limit

When $E_J/E_C \gg 40$, the charge dispersion is negligible and we can set $n_g = 0$ without loss.
The transmon Hamiltonian reduces to that of a **weakly anharmonic oscillator**:

$$\hat{H}_{\rm trans} \approx \omega_{01}\hat{a}^\dagger\hat{a} - \frac{E_C}{2}\hat{a}^{\dagger 2}\hat{a}^2,$$

with

$$\omega_{01} \approx \sqrt{8E_JE_C} - E_C, \qquad \alpha \approx -E_C.$$

**Physical interpretation of $K = E_C$.**
The Kerr term $-\frac{K}{2}\hat{n}(\hat{n}-1)$ means that each additional excitation lowers the effective
oscillator frequency by $K$.
This is equivalent to a cross-Kerr with itself — a self-Kerr or self-phase modulation.
The coupling to the cavity (Module 1c) generates a cross-Kerr $\chi$ between the qubit and cavity,
which is the mechanism behind dispersive readout.

**Why $n_g = 0$ (or the sweet spot)?**
At $n_g = 0$, the charge sensitivity $\partial\omega_{01}/\partial n_g = 0$ to all orders in $n_g$
(the spectrum has a maximum there for the qubit level).
Since transmon charge dispersion is already exponentially small, the choice of $n_g$
barely matters, but setting $n_g = 0$ simplifies the Hamiltonian.

**Higher-order corrections.** The exact Josephson cosine gives corrections to the Kerr model:

$$\hat{H} = \omega_{01}\hat{a}^\dagger\hat{a} - \frac{K}{2}\hat{a}^{\dagger 2}\hat{a}^2
+ \frac{K'}{6}\hat{a}^{\dagger 3}\hat{a}^3 - \cdots$$

These higher-order Kerr terms matter for high-excitation protocols (e.g., SNAP gates addressing $|n > 3\rangle$).

> **Lab note.** *The Kerr constant $K = E_C$ is directly related to the self-Kerr measured in pump-probe
> experiments: driving the qubit with a coherent state $|\alpha\rangle$ causes the IQ trajectory to rotate
> in a circle (Kerr rotation) at rate $K|\alpha|^2$.
> This is called the **AC Stark shift** and is used to calibrate $K$ in the lab.
> Typical values: $K/2\pi \approx 100$–$300\,\text{MHz}$ for transmons,
> $K/2\pi \approx 1$–$10\,\text{MHz}$ for the cavity field in Module 3.*


In [ ]:
# Transmon limit: verify alpha -> -Ec as Ej/Ec increases
ratios_lim = np.logspace(1, 2.5, 30)  # 10 to ~316
Ec_lim = Ec  # use global Ec parameter

alpha_vals = []
for r in ratios_lim:
    Ej_r = r * Ec_lim
    H_r = transmon_hamiltonian(Ej_r, Ec_lim, N)
    ev = H_r.eigenenergies()[:3]
    a = ((ev[2] - ev[1]) - (ev[1] - ev[0])) / (2 * np.pi)
    alpha_vals.append(a)

alpha_arr_lim = np.array(alpha_vals)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(ratios_lim, alpha_arr_lim, lw=2.5, color="steelblue", label=r"Numerical $\alpha$")
ax.axhline(-Ec_lim, ls="--", lw=1.8, color="tomato",
           label=rf"Transmon limit $-E_C = {-Ec_lim}$ GHz")
ax.set_xlabel(r"$E_J / E_C$", fontsize=12)
ax.set_ylabel(r"Anharmonicity $\alpha / 2\pi$ (GHz)", fontsize=12)
ax.set_title(r"Anharmonicity approaches $-E_C$ in the transmon limit", fontsize=11)
ax.legend()
ax.set_ylim(-0.35, 0.05)
plt.tight_layout()
plt.show()

# Default parameter check
H0 = transmon_hamiltonian(Ej, Ec, N)
ev0 = H0.eigenenergies()[:3]
alpha0 = ((ev0[2] - ev0[1]) - (ev0[1] - ev0[0])) / (2 * np.pi)
print(f"Default parameters: Ej={Ej}, Ec={Ec}, Ej/Ec={Ej/Ec:.0f}")
print(f"  alpha / 2pi = {alpha0:.4f} GHz")
print(f"  -Ec         = {-Ec:.4f} GHz")
print(f"  relative error |alpha + Ec| / Ec = {abs(alpha0 + Ec)/Ec:.4f}")
